# 04b Distillation results: the headline number
Reads the saved runs from `src/distill.py` (45 fits: 3 students x 3 arms x 5 seeds) and answers the question: how much of the teacher's gain does each student keep?

**Share of the teacher's gain kept** = (control RMSE − distilled RMSE) / (control RMSE − teacher RMSE). 0% means no better than the control, 100% means as good as the teacher.

The interval is a paired bootstrap over the test rows: draw the test rows with replacement 1,000 times, recompute every model's RMSE (averaged over its 5 seeds) and the ratio on each draw, and take the 2.5th and 97.5th percentiles. The seed-to-seed spread is reported alongside it.

In [1]:
import json

import numpy as np
import pandas as pd

from src.distill import ARMS, build_teacher
from src.lib import RESULTS_DIR, load_data, load_split, paired_bootstrap_rmse
from src.utils import load_config

cfg = load_config()
seed = cfg["seed"]
runs = pd.read_csv(RESULTS_DIR / "distill_runs.csv")
sq = np.load(RESULTS_DIR / "distill_sq_errors.npy")  # row i belongs to runs row i
teacher_rmse = json.loads((RESULTS_DIR / "distill_teacher.json").read_text())["teacher_test_rmse"]
print(f"{len(runs)} fits; per-row squared errors {sq.shape}; teacher test RMSE {teacher_rmse:.4f}")

45 fits; per-row squared errors (45, 4128); teacher test RMSE 0.4578


## The teacher's per-row errors

In [2]:
# distill.py saved the students' per-row errors but not the teacher's, so refit the same teacher.
X, y = load_data()
X, y = X.to_numpy(), y.to_numpy()
test_idx, folds = load_split()
dev_idx = np.sort(np.concatenate([val for _, val in folds]))
teacher = build_teacher(seed).fit(X[dev_idx], y[dev_idx])
sq_teacher = (teacher.predict(X[test_idx]) - y[test_idx]) ** 2
assert abs(np.sqrt(sq_teacher.mean()) - teacher_rmse) < 1e-9  # same teacher as in distill.py

## Test RMSE per student and arm
Mean and std over the 5 seeds.

In [3]:
students = list(runs["student"].unique())
stats = runs.groupby(["student", "arm"])["test_rmse"].agg(["mean", "std"])
stats.round(4)

mean     std
student        arm                                    
MLP 2x128      1 control                0.5200  0.0052
               2 distilled              0.5159  0.0025
               3 distilled + synthetic  0.4815  0.0034
MLP 2x32       1 control                0.5330  0.0094
               2 distilled              0.5318  0.0033
               3 distilled + synthetic  0.5068  0.0053
Tree (depth 6) 1 control                0.7052  0.0000
               2 distilled              0.7025  0.0000
               3 distilled + synthetic  0.7044  0.0040

## The headline: share of the teacher's gain kept, with an interval
Split into where the gain came from: teacher labels on the same rows (arm 2 vs. 1), extra synthetic rows (arm 3 vs. 2), and both together (arm 3 vs. 1). The RMSE drop is shown too, in units of $100k.

In [4]:
# 1,000 draws of the test rows with replacement. The same draw is used for every model.
groups = {
    f"{s}|{a}": sq[((runs["student"] == s) & (runs["arm"] == a)).to_numpy()]
    for s in students
    for a in ARMS
}
boot = paired_bootstrap_rmse(groups, sq_teacher, n_boot=1000, seed=seed)


def ci(x):
    return np.percentile(x, [2.5, 97.5])


rows = []
for s in students:
    control, labels, synth = (boot[f"{s}|{a}"] for a in ARMS)
    mean = {a: stats.loc[(s, a), "mean"] for a in ARMS}
    gain_draws = control - boot["teacher"]  # the teacher's gain over the control, per draw
    gain = mean[ARMS[0]] - teacher_rmse
    defined = ci(gain_draws)[0] > 0  # the ratio only makes sense if the gain is clearly above zero
    for name, better, worse_draws, better_draws, worse_mean, better_mean in [
        ("teacher labels (arm 2 vs. 1)", ARMS[1], control, labels, mean[ARMS[0]], mean[ARMS[1]]),
        ("synthetic rows (arm 3 vs. 2)", ARMS[2], labels, synth, mean[ARMS[1]], mean[ARMS[2]]),
        ("both (arm 3 vs. 1)", ARMS[2], control, synth, mean[ARMS[0]], mean[ARMS[2]]),
    ]:
        diff_draws = worse_draws - better_draws  # RMSE removed; positive means it helped
        share_draws = diff_draws / gain_draws
        rows.append(
            {
                "student": s,
                "effect": name,
                "rmse_drop": worse_mean - better_mean,
                "rmse_drop_lo": ci(diff_draws)[0],
                "rmse_drop_hi": ci(diff_draws)[1],
                "share_of_teacher_gain": (worse_mean - better_mean) / gain,
                "share_lo": ci(share_draws)[0],
                "share_hi": ci(share_draws)[1],
                "teacher_gain_over_control": gain,
                "teacher_gain_defined": bool(defined),
            }
        )
effects = pd.DataFrame(rows)
effects.to_csv(RESULTS_DIR / "distill_summary.csv", index=False)
print("teacher gain over the control is clearly above zero for:", effects.groupby("student")["teacher_gain_defined"].all().to_dict())

teacher gain over the control is clearly above zero for: {'MLP 2x128': True, 'MLP 2x32': True, 'Tree (depth 6)': True}


In [5]:
def fmt(r, lo, hi, pct):
    f = (lambda v: f"{v:.1%}") if pct else (lambda v: f"{v:.4f}")
    return f"{f(r[0])}  [{f(r[lo])}, {f(r[hi])}]"


view = pd.DataFrame(
    {
        "student": effects["student"],
        "effect": effects["effect"],
        "RMSE drop [95% CI]": [fmt((r.rmse_drop, r.rmse_drop_lo, r.rmse_drop_hi), 1, 2, False) for r in effects.itertuples()],
        "share of teacher's gain kept [95% CI]": [fmt((r.share_of_teacher_gain, r.share_lo, r.share_hi), 1, 2, True) for r in effects.itertuples()],
    }
).set_index(["student", "effect"])
view

RMSE drop [95% CI]  \
student        effect                                                     
MLP 2x32       teacher labels (arm 2 vs. 1)   0.0012  [-0.0051, 0.0076]   
               synthetic rows (arm 3 vs. 2)    0.0251  [0.0206, 0.0299]   
               both (arm 3 vs. 1)              0.0263  [0.0196, 0.0330]   
MLP 2x128      teacher labels (arm 2 vs. 1)   0.0041  [-0.0025, 0.0112]   
               synthetic rows (arm 3 vs. 2)    0.0344  [0.0290, 0.0394]   
               both (arm 3 vs. 1)              0.0386  [0.0315, 0.0466]   
Tree (depth 6) teacher labels (arm 2 vs. 1)   0.0027  [-0.0061, 0.0125]   
               synthetic rows (arm 3 vs. 2)  -0.0019  [-0.0098, 0.0052]   
               both (arm 3 vs. 1)             0.0007  [-0.0070, 0.0083]   

                                            share of teacher's gain kept [95% CI]  
student        effect                                                              
MLP 2x32       teacher labels (arm 2 vs. 1)                   1.6%  [-7.2%, 9.4%]  
               synthetic rows (arm 3 vs. 2)                 33.3%  [27.7%, 40.0%]  
               both (arm 3 vs. 1)                           34.9%  [28.6%, 40.4%]  
MLP 2x128      teacher labels (arm 2 vs. 1)                  6.7%  [-4.6%, 16.8%]  
               synthetic rows (arm 3 vs. 2)                 55.4%  [45.8%, 66.1%]  
               both (arm 3 vs. 1)                           62.0%  [55.5%, 68.2%]  
Tree (depth 6) teacher labels (arm 2 vs. 1)                   1.1%  [-2.5%, 5.0%]  
               synthetic rows (arm 3 vs. 2)                  -0.8%  [-4.0%, 2.2%]  
               both (arm 3 vs. 1)                             0.3%  [-2.9%, 3.3%]

### Findings
- **The headline.** Trained on the teacher's labels plus synthetic rows, the MLP with 2 layers of 128 keeps **62%** of the teacher's gain over the control (95% interval 56% to 68%), and the MLP with 2 layers of 32 keeps **35%** (29% to 40%). The depth-6 tree keeps **0.3%** (-2.9% to 3.3%), which is nothing. The teacher's gain over the control is clearly above zero for all three, so the ratio is defined.
- **Teacher labels alone do nothing measurable.** Arm 2 against arm 1 moves RMSE by 0.001 to 0.004, and every interval includes zero (for example 1.6%, from -7.2% to 9.4%, for the 32-wide MLP). Copying the teacher's answers on the same rows does not make a small model more accurate.
- **The extra synthetic rows carry the gain.** Arm 3 against arm 2 removes 0.025 RMSE for the 32-wide MLP and 0.034 for the 128-wide one, with intervals far from zero (33% and 55% of the teacher's gain). In units of $100k that is about $2,500 and $3,400.
- **The tree cannot use either.** Neither teacher labels nor synthetic rows help a depth-6 tree, in line with its poor fidelity to the teacher.
- **What the interval covers.** It reflects which test rows were drawn. The seed-to-seed std (0.003 to 0.009) is the same size as the arm 2 effects, which is another reason to read them as zero. This is one dataset with fixed student settings.
- **A limit on the explanation.** There is no arm with extra rows labeled by the truth, because synthetic rows have no true labels, so this cannot say whether any labels on more rows would help as much. It says the teacher's function sampled in more places is what lets a small MLP catch up.